# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwasay45/flyrankinternship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane:** Refresh / Content Opportunity Scoring

**ML task type:** Ranking / Scoring (with an optional classification view)

The core question is “Which pages should an editor review first for content refresh?”  
That is a ranking problem: we need an ordered list, not just a yes/no label.  

We can also train a classifier that predicts “is this page declining?” and then rank pages by the model’s predicted probability. The final output the editor uses is still a ranked review queue (score + reason codes + suggested action).

This matches the framing skill table:  
“Which ones first?” → Ranking / scoring → priority score → Precision@K.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
**Target / proxy:** `is_declining_label`  
(1 when `trend_direction == "down"`, else 0)

**Where it comes from:**  
It is derived from an **observed outcome** in the data: the change in impressions between the most recent 30 days and the previous 30 days (`trend_pct` → `trend_direction`).  

It is **not** a product rule or a health score. It is measured from real search traffic.

**Important honesty note:**  
In the starter data this label is built from the same 90-day window as many of the features. In the full warehouse we will move to a proper past-feature → future-label design so the target is truly “what happened next.” For framing purposes the starter label is still a valid observed proxy.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*
**Primary success metric:** Precision@50

**Why this metric:**  
An editor can realistically review only a limited number of pages (e.g. top 20–50).  
Precision@50 answers: “Of the 50 pages I put at the top of the queue, how many are actually declining / worth reviewing?”

**What “good” looks like:**  
- Hand-written rule in the starter pipeline: ~0.24 (about 12 of the top 50 correct)  
- Learned model already beats it by roughly 3× on the same metric  
- Goal for the lane: clearly beat a transparent baseline under client-holdout validation, while remaining readable.

Secondary metrics we will also report: Precision@20 and comparison against the baseline rule.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os, sys, subprocess

# --- Robust path setup (Colab + local) ---
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/abdulwasay45/flyrankinternship.git"
    REPO_DIR = "flyrankinternship"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create the target column exactly as the pipeline does
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print(f"Unit of analysis: one row = one page (content item)")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Declining rate: {df['is_declining_label'].mean():.1%}")
print()
print("Sample of the unit of analysis (key columns):")
display_cols = [
    "content_id", "client_id", "content_type",
    "impressions_90d", "days_since_last_update", "avg_position", "ctr",
    "trend_direction", "is_declining_label"
]
df[display_cols].head(8)

Working directory: /content/flyrankinternship/flyrankinternship
Unit of analysis: one row = one page (content item)
Shape: 30,000 rows × 45 columns
Declining rate: 54.2%

Sample of the unit of analysis (key columns):


,content_id,client_id,content_type,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,20,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,25,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,20,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,22,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,14,44.0,0.13,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,20,8.5,0.03,down,1
6,content_9a34b442b552,client_8722616204,keyword article,20,20,7.0,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,22,21.2,0.06,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
A fixed rule such as “stale (≥180 days) AND visible (≥500 impressions)” is transparent and useful, but it only captures one pattern.

Real pages show many overlapping signals at the same time:
- content age and days since last update
- current position and position tier
- impression volume and CTR
- engagement and scroll behaviour
- content type / intent

These signals interact. A page that is only moderately stale but has a sharp CTR drop and high volume may be more urgent than a very old page with almost no traffic. No short if-statement can encode all of those combinations cleanly.

A simple, readable model (e.g. shallow decision tree or logistic regression) can learn the relative importance of the signals from the data, produce a continuous priority score, and still be inspected by a human. That is why ranking / scoring with ML is the right task type here — not because we want a black box, but because the pattern is real and too messy for a single hand-written rule.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.